## Task 4: Context-Aware Chatbot (LangChain & RAG)
Problem Statement & Objective: Create an AI assistant capable of retrieving external information (RAG) and maintaining conversation history (Context Memory).

Dataset Loading & Preprocessing: Ingested Wikipedia data about Artificial Intelligence. Used RecursiveCharacterTextSplitter to create chunks and stored them in a FAISS vector database.

Model Development & Training: Integrated Google Gemini with LangChain. Used ConversationalRetrievalChain with ConversationBufferMemory to enable context tracking.

Evaluation Metrics: Evaluated based on retrieval relevance (RAG) and the ability of the model to resolve pronouns (Memory) in follow-up questions.
Visualizations: Live chat interface deployed via Streamlit.

Final Summary / Insights: RAG solves the problem of LLM hallucinations by grounding answers in real-time data, while memory creates a seamless, human-like conversational experience.

In [ ]:
# 1. Clean uninstall to remove broken versions
!pip uninstall -y langchain langchain-community langchain-huggingface langchain-core

# 2. Reinstall exact 2025 versions
!pip install -U langchain==0.3.0 langchain-community==0.3.0 langchain-huggingface==0.1.0 langchain-google-genai==2.0.0 langchain-text-splitters faiss-cpu sentence-transformers streamlit pyngrok

Found existing installation: langchain 0.3.0
Uninstalling langchain-0.3.0:
  Successfully uninstalled langchain-0.3.0
Found existing installation: langchain-community 0.3.0
Uninstalling langchain-community-0.3.0:
  Successfully uninstalled langchain-community-0.3.0
Found existing installation: langchain-huggingface 0.1.0
Uninstalling langchain-huggingface-0.1.0:
  Successfully uninstalled langchain-huggingface-0.1.0
Found existing installation: langchain-core 0.3.63
Uninstalling langchain-core-0.3.63:
  Successfully uninstalled langchain-core-0.3.63
  Using cached langchain-0.3.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_community-0.3.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached langchain_huggingface-0.1.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached langchain_text_splitters-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_core-0.3.81-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadat

In [ ]:
%%writefile app.py
import streamlit as st
import os
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

st.set_page_config(page_title="Task 4: RAG Chatbot", layout="centered")
st.title("🤖 Context-Aware AI (Task 4)")

# Sidebar for API Key
api_key = st.sidebar.text_input("Enter Gemini API Key", type="password")

if api_key:
    os.environ["GOOGLE_API_KEY"] = api_key

    try:
        # 1. Load Embeddings
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

        # 2. Load the Vector Store we created earlier
        if os.path.exists("faiss_ai_index"):
            vector_db = FAISS.load_local("faiss_ai_index", embeddings, allow_dangerous_deserialization=True)

            # 3. Setup Memory (Requirement: Context History)
            if "memory" not in st.session_state:
                st.session_state.memory = ConversationBufferMemory(
                    memory_key="chat_history",
                    return_messages=True
                )

            # 4. Initialize LLM (Using your verified available model)
            llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

            # 5. Build RAG Chain
            qa_chain = ConversationalRetrievalChain.from_llm(
                llm=llm,
                retriever=vector_db.as_retriever(),
                memory=st.session_state.memory
            )

            # Chat UI Logic
            if "messages" not in st.session_state:
                st.session_state.messages = []

            for message in st.session_state.messages:
                with st.chat_message(message["role"]):
                    st.markdown(message["content"])

            if prompt := st.chat_input("Ask about the documents..."):
                st.session_state.messages.append({"role": "user", "content": prompt})
                with st.chat_message("user"):
                    st.markdown(prompt)

                with st.chat_message("assistant"):
                    # The chain retrieves context, uses memory, and generates an answer
                    response = qa_chain.invoke({"question": prompt})
                    answer = response['answer']
                    st.markdown(answer)
                    st.session_state.messages.append({"role": "assistant", "content": answer})
        else:
            st.error("Vector index 'faiss_ai_index' not found. Please run the Vectorization cell in Colab first.")
    except Exception as e:
        st.error(f"Error: {e}")
else:
    st.info("Please enter your Gemini API Key in the sidebar.")

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import threading
import time

# 1. Kill any zombie processes
!pkill streamlit
!pkill ngrok

# 2. Configure Ngrok
ngrok.set_auth_token("35c1sNj52z3eAfvg7Pdj1hV1xiT_7JQnM1NZ9WXZvCiE6YuME")

# 3. Start Streamlit using the Python Module flag (Fixes ModuleNotFoundError)
def run_streamlit():
    !python3 -m streamlit run app.py --server.port 8501 --server.enableCORS=false

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# 4. Generate Link
time.sleep(10)
public_url = ngrok.connect(8501).public_url
print(f"\n✅ YOUR CHATBOT IS READY!")
print(f"CLICK THIS LINK: {public_url}")

2025-12-29 19:07:07.029 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            


2025-12-29 19:07:08.941 Port 8501 is already in use

✅ YOUR CHATBOT IS READY!
CLICK THIS LINK: https://d829e3004945.ngrok-free.app
